In [1]:
import radiate as rd
import polars as pl

rd.random.seed(67123)

### Create a fake dataset 

This dataset is a simple polynomial regression problem with a small it of noise

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
target_species = 4
species_threshold = 0.2

rolling_species_count = rd.Expr.select("species.count").rolling(25).mean()
err = rd.Expr.select("species.count").error(target_species) * 0.01

### Create the engine

This engine evolves `rd.Graph`s to fit the polynomial regression problem. The vertices of the graph are only allowed to use addition, multiplication, and constants. The engine also tracks the species count and species threshold as metrics, and uses the NEAT distance as a diversity metric.

In [4]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.sum()],
        edge=rd.Op.weight(),
        output=rd.Op.sum(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .diversity(
        rd.Dist.neat(),
        threshold=species_threshold,
        target=target_species,
    )
    .metrics(
        species_rolling=rolling_species_count,
        error=err,
    )
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-10-17T12:15:03.550072Z  INFO State Change: PreStart -> Running
2026-10-17T12:15:03.553717Z  INFO Epoch 1    | Score:   2.0038 | Time: 3.36ms
2026-10-17T12:15:03.554489Z  INFO Epoch 2    | Score:   2.0038 | Time: 3.76ms
2026-10-17T12:15:03.554635Z  INFO Epoch 3    | Score:   1.6821 | Time: 3.89ms
2026-10-17T12:15:03.554740Z  INFO Epoch 4    | Score:   1.6821 | Time: 3.98ms
2026-10-17T12:15:03.554850Z  INFO Epoch 5    | Score:   1.6821 | Time: 4.07ms
2026-10-17T12:15:03.554946Z  INFO Epoch 6    | Score:   1.6821 | Time: 4.16ms
2026-10-17T12:15:03.555057Z  INFO Epoch 7    | Score:   1.6821 | Time: 4.26ms
2026-10-17T12:15:03.555160Z  INFO Epoch 8    | Score:   1.6821 | Time: 4.35ms
2026-10-17T12:15:03.555265Z  INFO Epoch 9    | Score:   1.5757 | Time: 4.44ms
2026-10-17T12:15:03.555378Z  INFO Epoch 10   | Score:   1.5757 | Time: 4.54ms
2026-10-17T12:15:03.555546Z  INFO Epoch 11   | Score:   1.5757 | Time: 4.67ms
2026-10-17T12:15:03.555685Z  INFO Epoch 12   | Score:   1.5757 | Time: 4.7

### Metrics

Using the `rd.MetricCollector` we can aggregate the metrics for the whole engine's run and visualize the metrics we want to track here once the engine has finished. In this case we only want to see our three expressions, but we could also see any `rd.Metric` that the engine tracks - refer to the docs for more details.

In [5]:
collector.plot("species.count", "species.threshold", "error")

In [6]:
df = collector.to_polars()
df.filter(pl.col("name") == "species.count").tail(20)
# df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""species.count""",4.0,2268.0,4.101266,1.016467,1.033205,871.898071,2.0,17.0,553,null,null,null,null,null,null,552,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2272.0,4.101083,1.015556,1.031355,953.270203,2.0,17.0,554,null,null,null,null,null,null,553,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2276.0,4.100901,1.014649,1.029512,1048.008667,2.0,17.0,555,null,null,null,null,null,null,554,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2280.0,4.100719,1.013743,1.027675,1159.380371,2.0,17.0,556,null,null,null,null,null,null,555,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2284.0,4.100539,1.01284,1.025845,1291.791016,2.0,17.0,557,null,null,null,null,null,null,556,1,"[""species"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""species.count""",4.0,2328.0,4.098591,1.003063,1.006136,16358.220703,2.0,17.0,568,null,null,null,null,null,null,567,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2332.0,4.098418,1.002188,1.004381,35187.042969,2.0,17.0,569,null,null,null,null,null,null,568,1,"[""species"", ""statistic""]"
"""species.count""",4.0,2336.0,4.098246,1.001316,1.002633,181912.9375,2.0,17.0,570,null,null,null,null,null,null,569,1,"[""species"", ""statistic""]"
